# MatCore yabadaba example

This provides information for using yabadaba to interact with records in a database.

Examples are low as I don't feel like generating demo MatCore records...

In [1]:
import pandas as pd

# Import core yabadaba and extension package
import yabadaba
import yabadaba_matcore

## 1. Database basics

yabadaba provides common database interaction methods that work on different database infrastructures.

### 1.1. Supported database infrastructures

- mongo: MongoDB based on JSON (BSON) records.
- cdcs: Materials data curation system database based on XML records.
- local: A local directory of JSON or XML files, plus auto-generated CSV cache files.  Allows database operations without any actual "database".

### 1.2. Basic record interaction methods

- get_records() returns a numpy array of Record objects of a given style.  Recognizes query terms (see below) for filtering which results are returned.  Also, has a return_df setting that returns a pandas DataFrame object of the associated record metadata() fields.
- get_records_df() is like above with return_df=True, except that only the DataFrame is returned and not the records.
- get_record() returns exactly one matching Record object.  Will throw an error if exactly one match is not found.
- count_records() returns only the count of matching records.
- retrieve_record() gets a single matching record from the database and saves it.
- add_record() adds a new record to the database.
- update_record() updates the contents of a record already in the database.
- delete_record() deletes a record from the database.

### 1.3. Raw data files

Raw data files can also stored in the database alongside the records.  The default yabadaba behavior is to store all data files associated with a record in a single tar.gz file.

- get_tar() returns a record's tar contents as a TarFile object.
- add_tar() adds a new tar file associated with a record to the database.  Can either give a tar object or a path to a folder to archive.
- update_tar() replaces a record's tar file with a new one.
- delete_tar() deletes a record's tar file from the database.

Record objects also have convenience methods and attributes supporting easier file interactions.

- Record.database is a database object where the record is expected to be.  This is automatically set if you used one of the get_record methods to get the record.
- Record.tar is a TarFile object of the record's tar contents.  The tar contents are automatically retrieved from the database the first time tar is accessed.
- Record.clear_tar() closes the tarfile and resets the attribute.  This should hopefully allow for Python to flush it from memory.
- Record.get_file() retrieves a single file from the record's tar by name and returns it as an io object.
- Record.display_image() works for IPython environments where it retrieves an image file from the record's tar and displays it.


### 1.4. Advanced methods

- copy_records() copies records from one database to another with overwrite and includetar options.  Can either give a list of records or a record style.
- destroy_records() deletes all specified records and tar files from the database.  Can either give a list of records or a record style.

## 2. Query operations

Most record value styles have a default query operation associated with them.  These operations are automatically collected and recognized as keyword parameters for the get_record methods.  You can view the recognized terms using the record's querydoc.

__Caution #1:__ The queries are collected in a dict based on the parameter names.  If values in a subset are named similar to values elsewhere, then the query operation may be replaced.  I was too lazy to check here, but any conflicts can be avoided by modifying or removing queries.

__Caution #2:__ The 'local' database style only supports querying terms up to two layers deep, i.e. only primary and subset terms.  Deeper queries likely work for the other styles, but hasn't been fully tested.

In [2]:
yabadaba.querydoc('matcore', render=True)

# matcore Query Parameters

- __name__ (*str or list, optional*): Return only the records where The name of the author who generated the data. matches a given value
- __affiliation__ (*str or list, optional*): Return only the records where The affiliation of the author who generated the data. matches a given value
- __title__ (*str or list, optional*): Return only the records where A single sentence description of the dataset. matches a given value
- __creation_date__ (*str or list, optional*): Return only the records where The calendar date when the dataset was created. matches a given value
- __description__ (*str or list, optional*): Return only the records where Explanation of the relationship between the related content and the current dataset. matches a given value
- __disclaimer__ (*str or list, optional*): Return only the records where A statement of applicability provided by the creator(s) informing users of the intended use and/or limitations of this dataset. matches a given value
- __phase__ (*str or list, optional*): Return only the records where The structure of the material included in the dataset. matches a given value
- __microstructure__ (*str or list, optional*): Return only the records where The arrangement of phases, grains and defects in a material. matches a given value
- __method_class__ (*str or list, optional*): Return only the records where The general category to which the computational technique belongs. matches a given value
- __method__ (*str or list, optional*): Return only the records where The computational materials science (CMS) approach used in the computation. matches a given value
- __type__ (*str or list, optional*): Return only the records where The nature of the simulation being performed, whether equilibrium, nonequilibrium, or nonstandard. matches a given value
- __number_of_particles__ (*int, tuple or list, optional*): Return only the records where A count of atoms or other discrete entities comprising the material. matches a given value or is in a given range
- __volume__ (*float, tuple or list, optional*): Return only the records where The amount of space occupied by the material. matches a given value or is in a given range
- __mass_density__ (*float, tuple or list, optional*): Return only the records where The total number of particles comprising the material divided by the volume it occupies. matches a given value or is in a given range
- __temperature__ (*float, tuple or list, optional*): Return only the records where A measure of the hotness or coldness of an external heat bath in thermal contact with the material. matches a given value or is in a given range
- __reference__ (*str or list, optional*): Return only the records where Text that uniquely identifies the source of information being cited. matches a given value
- __doi__ (*str or list, optional*): Return only the records where The digital object identifier (DOI) for the source. matches a given value
- __link__ (*str or list, optional*): Return only the records where A URI pointing to a permanent location of the source. matches a given value
- __award_title__ (*str or list, optional*): Return only the records where Name of the grant that provided funding to generate the dataset. matches a given value
- __funder__ (*str or list, optional*): Return only the records where The name of the funding agency that provided money and/or resources to generate the dataset. matches a given value
- __award_number__ (*str or list, optional*): Return only the records where A funder identifier for the grant. matches a given value
- __links__ (*str or list, optional*): Return only the records where A list of permanent pointers to related datasets (such as MatCore IDs, DOIs, URIs, etc.) contains a given value
- __event_type__ (*str or list, optional*): Return only the records where Description of change made to the dataset. matches a given value
- __date__ (*str or list, optional*): Return only the records where The data when the change was made. matches a given value
- __agent__ (*str or list, optional*): Return only the records where Identity of the entity responsible for the change. matches a given value
- __comments__ (*str or list, optional*): Return only the records where Explanation for the change in provenance. matches a given value
- __checksum__ (*str or list, optional*): Return only the records where A digital fingerprint for the dataset of associated files. matches a given value
- __matcore_id__ (*str or list, optional*): Return only the records where An identifier for the dataset. matches a given value
- __matcore_date__ (*str or list, optional*): Return only the records where The calendar date when this MatCore document was created. matches a given value
- __license__ (*str or list, optional*): Return only the records where A contract defining the terms and conditions under which the dataset can be used. matches a given value


## 3. Query example

I'm using iprPy here, which builds on yabadaba, as an example.  If I had MatCore records I could set up a more direct example.

Mentally, just replace "iprPy" below with "yabadaba" as they are essentially calling the same operations.

In [3]:
import iprPy

In [4]:
# Load database using pre-defined settings stored as 'potentials'
database = iprPy.load_database('potentials')
print(database)

database style cdcs at https://potentials.nist.gov/


In [5]:
yabadaba.querydoc('relaxed_crystal', render=True)

# relaxed_crystal Query Parameters

- __key__ (*str or list, optional*): Return only the records where key matches a given value
- __url__ (*str or list, optional*): Return only the records where url matches a given value
- __method__ (*str or list, optional*): Return only the records where method matches a given value
- __standing__ (*str or list, optional*): Return only the records where standing matches a given value
- __potential_LAMMPS_key__ (*str or list, optional*): Return only the records where potential_LAMMPS_key matches a given value
- __potential_LAMMPS_id__ (*str or list, optional*): Return only the records where potential_LAMMPS_id matches a given value
- __potential_LAMMPS_url__ (*str or list, optional*): Return only the records where potential_LAMMPS_url matches a given value
- __potential_key__ (*str or list, optional*): Return only the records where potential_key matches a given value
- __potential_id__ (*str or list, optional*): Return only the records where potential_id matches a given value
- __potential_url__ (*str or list, optional*): Return only the records where potential_url matches a given value
- __temperature__ (*float, tuple or list, optional*): Return only the records where temperature matches a given value or is in a given range
- __pressure_xx__ (*float, tuple or list, optional*): Return only the records where pressure_xx matches a given value or is in a given range
- __pressure_yy__ (*float, tuple or list, optional*): Return only the records where pressure_yy matches a given value or is in a given range
- __pressure_zz__ (*float, tuple or list, optional*): Return only the records where pressure_zz matches a given value or is in a given range
- __pressure_xy__ (*float, tuple or list, optional*): Return only the records where pressure_xy matches a given value or is in a given range
- __pressure_xz__ (*float, tuple or list, optional*): Return only the records where pressure_xz matches a given value or is in a given range
- __pressure_yz__ (*float, tuple or list, optional*): Return only the records where pressure_yz matches a given value or is in a given range
- __family__ (*str or list, optional*): Return only the records where family matches a given value
- __family_url__ (*str or list, optional*): Return only the records where family_url matches a given value
- __parent_key__ (*str or list, optional*): Return only the records where parent_key matches a given value
- __parent_url__ (*str or list, optional*): Return only the records where parent_url matches a given value
- __symbols__ (*str or list, optional*): Return only the records that contain the listed atomic symbols
- __composition__ (*str or list, optional*): Return only the records where composition matches a given value
- __crystalfamily__ (*str or list, optional*): Return only the records where crystalfamily matches a given value
- __natypes__ (*int, tuple or list, optional*): Return only the records where natypes matches a given value or is in a given range
- __a__ (*float, tuple or list, optional*): Return only the records where a matches a given value or is in a given range
- __b__ (*float, tuple or list, optional*): Return only the records where b matches a given value or is in a given range
- __c__ (*float, tuple or list, optional*): Return only the records where c matches a given value or is in a given range
- __alpha__ (*float, tuple or list, optional*): Return only the records where alpha matches a given value or is in a given range
- __beta__ (*float, tuple or list, optional*): Return only the records where beta matches a given value or is in a given range
- __gamma__ (*float, tuple or list, optional*): Return only the records where gamma matches a given value or is in a given range
- __natoms__ (*int, tuple or list, optional*): Return only the records where natoms matches a given value
- __potential_energy__ (*float, tuple or list, optional*): Return only the records where potential_energy matches a given value or is in a given range
- __cohesive_energy__ (*float, tuple or list, optional*): Return only the records where cohesive_energy matches a given value or is in a given range


Fetch all good relaxed crystal structures for fcc gold that remained fcc following a dynamic relaxation.

In [6]:
records, df = database.get_records('relaxed_crystal', method='dynamic', standing='good', family='A1--Cu--fcc', composition='Au', return_df=True)

100%|████████████████████████████████████████████████████████████████| 45/45 [00:03<00:00,  8.88it/s]


In [7]:
df.keys()

Index(['name', 'key', 'url', 'method', 'standing', 'potential_LAMMPS_key',
       'potential_LAMMPS_id', 'potential_LAMMPS_url', 'potential_key',
       'potential_id', 'potential_url', 'family', 'family_url', 'parent_key',
       'parent_url', 'symbols', 'composition', 'crystalfamily', 'natypes', 'a',
       'b', 'c', 'alpha', 'beta', 'gamma', 'natoms', 'Epot (eV/atom)',
       'Ecoh (eV/atom)'],
      dtype='object')

In [8]:
df[['potential_LAMMPS_id', 'a', 'Ecoh (eV/atom)']].sort_values(['potential_LAMMPS_id', 'a'])

,potential_LAMMPS_id,a,Ecoh (eV/atom)
28,1986--Foiles-S-M--Ag-Au-Cu-Ni-Pd-Pt--LAMMPS--ipr1,4.080000,-3.930000
11,1986--Foiles-S-M--Au--LAMMPS--ipr1,4.080000,-3.930000
29,1987--Ackland-G-J--Au--LAMMPS--ipr1,4.078001,-3.789045
30,1987--Ackland-G-J--Au--LAMMPS--ipr2,4.078001,-3.789045
1,1989--Adams-J-B--Ag-Au-Cu-Ni-Pd-Pt--LAMMPS--ipr1,4.080000,-3.930000
36,1989--Adams-J-B--Au--LAMMPS--ipr1,4.080000,-3.930000
7,1990--Ackland-G-J--Cu-Ag-Au--LAMMPS--ipr1,4.078000,-3.789045
32,2003--Lee-B-J--Au--LAMMPS--ipr1,4.072935,-3.930000
26,2004--Zhou-X-W--Au--LAMMPS--ipr1,4.079787,-3.930005
35,2004--Zhou-X-W--Au--LAMMPS--ipr2,4.080053,-3.930005
